# First Model Exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
train = pd.read_csv('data/train_processed.csv')

In [3]:
train.columns

Index(['Id', 'Slope', 'Soil_Type1', 'Soil_Type2', 'Soil_Type3', 'Soil_Type4',
       'Soil_Type5', 'Soil_Type6', 'Soil_Type7', 'Soil_Type8', 'Soil_Type9',
       'Soil_Type10', 'Soil_Type11', 'Soil_Type12', 'Soil_Type13',
       'Soil_Type14', 'Soil_Type16', 'Soil_Type17', 'Soil_Type18',
       'Soil_Type19', 'Soil_Type20', 'Soil_Type21', 'Soil_Type22',
       'Soil_Type23', 'Soil_Type24', 'Soil_Type25', 'Soil_Type26',
       'Soil_Type27', 'Soil_Type28', 'Soil_Type29', 'Soil_Type30',
       'Soil_Type31', 'Soil_Type32', 'Soil_Type33', 'Soil_Type34',
       'Soil_Type35', 'Soil_Type36', 'Soil_Type37', 'Soil_Type38',
       'Soil_Type39', 'Soil_Type40', 'Cover_Type', 'Distance_To_Hydrology',
       'Log_Horiz_Hydrology', 'Log_Horiz_Roadways', 'Log_Horiz_Fire',
       'Aspect_sin', 'Aspect_cos', 'Hillshade_Mean',
       'Elevation_Wilderness_Area1', 'Elevation_Wilderness_Area2',
       'Elevation_Wilderness_Area3', 'Elevation_Wilderness_Area4',
       'Close_To_Water', 'High_Altitude'],


In [4]:
X = train.drop(columns=["Cover_Type", "Id"])
y = train["Cover_Type"]
y_shifted = y - 1

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

from sklearn.neighbors import KNeighborsClassifier


from sklearn.tree import DecisionTreeClassifier


from sklearn.svm import LinearSVC, SVC

from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


In [8]:
basic_models = {
    "LogReg": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "LDA": Pipeline([("scaler", StandardScaler()), ("model", LinearDiscriminantAnalysis())]),
    "QDA": Pipeline([("scaler", StandardScaler()), ("model", QuadraticDiscriminantAnalysis())]),
    "Naive Bayes": GaussianNB(),
    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Linear SVM": Pipeline([("scaler", StandardScaler()), ("model", LinearSVC(dual=False, random_state=42))]),
    "Kernel SVM (RBF)": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", random_state=42))]),
    "Bagging (Trees)": BaggingClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
basic_results = []

for name, model in basic_models.items():
    print(f"Model: {name}")

    if any(x in name for x in ["XGB", "LightGBM"]):
        y_true = y_shifted
    else:
        y_true = y

    cv_results = cross_validate(model, X, y_true, cv=skf, scoring="accuracy", n_jobs=-1)
    
    basic_results.append({
        "Model": name,
        "Mean Accuracy": cv_results["test_score"].mean(),
        "Std Dev": cv_results["test_score"].std()
    })

Model: LogReg
Model: LDA
Model: QDA


/Users/benedikt/anaconda3/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/benedikt/anaconda3/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/benedikt/anaconda3/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/Users/benedikt/anaconda3/lib/python3.11/site-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help 

Model: Naive Bayes
Model: KNN
Model: Decision Tree
Model: Linear SVM
Model: Kernel SVM (RBF)
Model: Bagging (Trees)
Model: Random Forest
Model: AdaBoost
Model: Gradient Boosting
Model: XGBoost
Model: LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002516 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002479 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2718
[LightGBM] [Info] Total Bins 2718
[LightGBM] [Info] Number of data points in the train set: 12096, number of used features: 44
[LightGBM] [Info] Number of data points in the train set: 12096, number of used features: 44
[LightGBM] [Info] Start training from score -1.945910
[LightGBM] [Info] Start trainin

In [9]:
basic_results_df = pd.DataFrame(basic_results)
print(basic_results_df.sort_values(["Mean Accuracy"], ascending=False))

                Model  Mean Accuracy   Std Dev
9       Random Forest       0.860119  0.008199
13           LightGBM       0.859061  0.008803
12            XGBoost       0.858730  0.008050
8     Bagging (Trees)       0.845172  0.007261
11  Gradient Boosting       0.801389  0.010408
5       Decision Tree       0.794643  0.009360
4                 KNN       0.773413  0.003842
7    Kernel SVM (RBF)       0.713426  0.006916
0              LogReg       0.690873  0.006515
6          Linear SVM       0.674735  0.005806
1                 LDA       0.645238  0.007010
3         Naive Bayes       0.513624  0.006300
10           AdaBoost       0.467593  0.036825
2                 QDA       0.447751  0.015000


In [10]:
from sklearn.neural_network import MLPClassifier

In [14]:
nn_models = {
    "MLP": Pipeline([
        ("scaler", StandardScaler()), 
        ("model", MLPClassifier(max_iter=1000, random_state=42)) #  Increased max_iter to 1000 for convergence
    ]),
    "DNN": Pipeline([
        ("scaler", StandardScaler()), 
        ("model", MLPClassifier(hidden_layer_sizes=(100, 100, 100), max_iter=1000, random_state=42)) #  Increased max_iter to 1000 for convergence
    ])
}
nn_results = []

for name, model in nn_models.items():
    print(f"Model: {name}")
    cv_nn_results = cross_validate(model, X, y_true, cv=skf, scoring="accuracy", n_jobs=-1)

    nn_results.append({
        "Model": name,
        "Mean Accuracy": cv_nn_results["test_score"].mean(),
        "Std Dev": cv_nn_results["test_score"].std()
    })

Model: MLP
Model: DNN


In [15]:
nn_results_df = pd.DataFrame(nn_results)
print(nn_results_df.sort_values(["Mean Accuracy"], ascending=False))

  Model  Mean Accuracy   Std Dev
1   DNN       0.823214  0.004632
0   MLP       0.815079  0.011062
